## DATA LOADING AND DATA CLEANING

## 1. Import the required libraries

In [4]:
import pandas as pd
import numpy as np
import re

Pandas - loading and manipulating the CSV dataset.<br>
NumPy - numerical operations and handling missing values.<br>
re - text cleaning particularly removig unnecessary whitespace and formatting.<br>

## 2. Loading Dataset

In [5]:
df = pd.read_csv("../data/raw/merged_jobs_dataset.csv")
df.shape


(31321, 12)

The original dataset contains 31321 job postings(rows) and 12 variables(columns)

In [6]:
display(df.head())

,title,company,location,industry_category,work_type,salary_range,experience_level,min_qualification,description,requirements,source_url,source_platform
0,Marketing Intern,"We're Food52, and we've created a groundbreaki...","US, NY, New York",NaN,Other,NaN,Internship,NaN,"Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,NaN,Fake Job Postings Dataset
1,Customer Service - Cloud Video Production,"90 Seconds, the worlds Cloud Video Production ...","NZ, , Auckland",Marketing and Advertising,Full-time,NaN,Not Applicable,NaN,Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,NaN,Fake Job Postings Dataset
2,Commissioning Machinery Assistant (CMA),Valor Services provides Workforce Solutions th...,"US, IA, Wever",NaN,NaN,NaN,NaN,NaN,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,NaN,Fake Job Postings Dataset
3,Account Executive - Washington DC,Our passion for improving quality of life thro...,"US, DC, Washington",Computer Software,Full-time,NaN,Mid-Senior level,Bachelor's Degree,THE COMPANY: ESRI – Environmental Systems Rese...,"EDUCATION: Bachelor’s or Master’s in GIS, busi...",NaN,Fake Job Postings Dataset
4,Bill Review Manager,SpotSource Solutions LLC is a Global Human Cap...,"US, FL, Fort Worth",Hospital & Health Care,Full-time,NaN,Mid-Senior level,Bachelor's Degree,JOB TITLE: Itemization Review ManagerLOCATION:...,QUALIFICATIONS:RN license in the State of Texa...,NaN,Fake Job Postings Dataset


Displays the five records so that we can understand what the data looks like. 

### 3. Inspect the structure of the dataset

In [7]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 31321 entries, 0 to 31320
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   title              31318 non-null  str  
 1   company            22182 non-null  str  
 2   location           26769 non-null  str  
 3   industry_category  24914 non-null  str  
 4   work_type          21123 non-null  str  
 5   salary_range       8521 non-null   str  
 6   experience_level   12565 non-null  str  
 7   min_qualification  11510 non-null  str  
 8   description        29816 non-null  str  
 9   requirements       15184 non-null  str  
 10  source_url         13441 non-null  str  
 11  source_platform    31321 non-null  str  
dtypes: str(12)
memory usage: 2.9 MB
None


Number of records

In [8]:
print(df.columns)

Index(['title', 'company', 'location', 'industry_category', 'work_type',
       'salary_range', 'experience_level', 'min_qualification', 'description',
       'requirements', 'source_url', 'source_platform'],
      dtype='str')


column names

In [9]:
print(df.dtypes)

title                str
company              str
location             str
industry_category    str
work_type            str
salary_range         str
experience_level     str
min_qualification    str
description          str
requirements         str
source_url           str
source_platform      str
dtype: object


Data types

## 4. Check duplicate records

In [10]:
print("Number of duplicate records:", df.duplicated().sum())

Number of duplicate records: 324


In [11]:
df = df.drop_duplicates().copy()

print("Dataset shape after removing duplicates:", df.shape)

Dataset shape after removing duplicates: (30997, 12)


## 5. Check for missing values

In [12]:
missing_values = df.isnull().sum()

print(missing_values)

title                    3
company               9107
location              4547
industry_category     6336
work_type            10151
salary_range         22530
experience_level     18661
min_qualification    19709
description           1505
requirements         16086
source_url           17556
source_platform          0
dtype: int64


Checking the missing information

In [13]:
missing_summary = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
    "Missing_Percentage": (df.isnull().sum() / len(df) * 100).round(2)
})

print(missing_summary)

                   Missing_Count  Missing_Percentage
title                          3                0.01
company                     9107               29.38
location                    4547               14.67
industry_category           6336               20.44
work_type                  10151               32.75
salary_range               22530               72.68
experience_level           18661               60.20
min_qualification          19709               63.58
description                 1505                4.86
requirements               16086               51.90
source_url                 17556               56.64
source_platform                0                0.00


Calculating percentages of missing information

In [14]:
# Cleaning text columns
text_columns = [
    "title",
    "company",
    "location",
    "industry_category",
    "work_type",
    "salary_range",
    "experience_level",
    "min_qualification",
    "description",
    "requirements",
    "source_url",
    "source_platform"
]

for column in text_columns:
    df[column] = (
        df[column]
        .astype("string")  # converts columns to pandas string type
        .str.replace(r"\s+", " ", regex=True)  # Removes unnecessary white space
        .str.strip()
    )

In [15]:
# Standardizing
# employment type
df["work_type"] = df["work_type"].replace({
    "Full-Time": "Full-time",
    "Full time": "Full-time",
    "Part time": "Part-time",
    "Part-Time": "Part-time"
})

# experience level
df["experience_level"] = df["experience_level"].replace({
    "Mid-Senior level": "Mid-Senior",
    "Mid level": "Mid-level",
    "Senior level": "Senior",
    "Entry level": "Entry-level",
    "Executive level": "Executive",
    "Internship & Graduate": "Internship/Graduate"
})

# education qualifications
df["min_qualification"] = df["min_qualification"].replace({
    "Bachelors": "Bachelor's Degree",
    "Masters": "Master's Degree",
    "Highschool": "High School or equivalent"
})

In [16]:
# --------------------------------------------------
# HANDLE MISSING VALUES
# --------------------------------------------------

# 1. Remove records with missing/empty job titles
df = df[                 # removing the three job postings without titles
    df["title"].notna() &
    df["title"].str.strip().ne("")
].copy()

# 2. Categorical variables
categorical_columns = [     #Replacing missing categorical values with "unknown"
    "company",
    "location",
    "industry_category",
    "work_type",
    "experience_level",
    "min_qualification"
]

for column in categorical_columns:
    df[column] = df[column].fillna("Unknown")

# 3. Text variables    # Replacing missing text with an empty string 
df["description"] = df["description"].fillna("")
df["requirements"] = df["requirements"].fillna("")

# 4. Salary   #Replacing the missing salary range with "not stated" 
df["salary_range"] = df["salary_range"].fillna("Not stated")

# 5. Source URL  # replacing missing source URL with "Not available"
df["source_url"] = df["source_url"].fillna("Not available")

In [17]:
df.duplicated().sum()

np.int64(59)

After the standardizing steps catches the 59 rows that became identical because of the whitespace stripping, category standardization, and fillna placeholders applied.

In [18]:
df = df.drop_duplicates().copy()
print("Shape after final dedup:", df.shape)
print("Duplicate records remaining:", df.duplicated().sum())

Shape after final dedup: (30935, 12)
Duplicate records remaining: 0


In [19]:
print(df.isnull().sum())

title                0
company              0
location             0
industry_category    0
work_type            0
salary_range         0
experience_level     0
min_qualification    0
description          0
requirements         0
source_url           0
source_platform      0
dtype: int64


Checking if there is missing information after filling in missing values.

In [20]:
print(df.shape)

(30935, 12)


## 6. Check the cleaned dataset

In [21]:
print("Final dataset shape:")
print(df.shape)

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate records:")
print(df.duplicated().sum())

Final dataset shape:
(30935, 12)

Data types:
title                string
company              string
location             string
industry_category    string
work_type            string
salary_range         string
experience_level     string
min_qualification    string
description          string
requirements         string
source_url           string
source_platform      string
dtype: object

Missing values:
title                0
company              0
location             0
industry_category    0
work_type            0
salary_range         0
experience_level     0
min_qualification    0
description          0
requirements         0
source_url           0
source_platform      0
dtype: int64

Duplicate records:
0


## 7. Save the cleaned dataset

In [22]:
df.to_csv(
    "cleaned_job_scam_dataset.csv",
    index=False
)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.
